# SoulX-FlashHead 듀오 아바타 서버 (실험용, MuseTalk과 별개)

MuseTalk 대신 [SoulX-FlashHead](https://github.com/Soul-AILab/SoulX-FlashHead)로 립싱크 영상을 만드는 실험용 노트북입니다.
`musetalk_duo_avatar_server.ipynb`는 건드리지 않고, 여기서 실시간성이 A100/L4로 확보되는지 먼저 검증합니다.

**MuseTalk과의 핵심 차이**: MuseTalk은 실제 촬영 영상 위에 입 부분만 다시 그리는 후처리 방식이라 매 프레임
얼굴을 검출하면서 위치가 미세하게 흔들리는(떨림) 문제가 있었습니다. SoulX-FlashHead는 사진 한 장 + 오디오로
영상 전체(고개 움직임 포함)를 처음부터 생성하는 end-to-end 모델이라 이 떨림 문제가 구조적으로 없을 것으로
기대합니다. 다만 GPU 요구사항이 더 높고(공식 기준 RTX4090급), 저희 환경(A100/L4)에서 실시간이 나오는지는
아직 확인되지 않았습니다.

**Lite 모델로 시작합니다** — Pro 모델은 공식 벤치마크상 dual RTX5090이 있어야 실시간(25FPS+)이 나오고
단일 GPU에서는 10.8FPS라 A100/L4로는 부족할 가능성이 높습니다. Lite 모델은 단일 RTX4090에서 96FPS,
동시 3스트림도 25FPS+가 나온다고 하니 듀오(2명 동시) 용도에 더 적합합니다.


In [ ]:
# 1. GPU 확인 + Google Drive 마운트
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Miniconda 설치 + Python 3.10 가상환경 구성 (Drive에 캐시되면 다음 실행부터 빠름)
# MuseTalk 노트북과 완전히 다른 torch/CUDA 버전(torch 2.7.1 + CUDA 12.8)이 필요해서,
# 같은 miniconda 설치 안에 별도 이름(soulxflash)의 환경으로 분리합니다.
import os

CONDA_PREFIX = "/usr/local/miniconda"
ENV_NAME = "soulxflash"
CONDA_BIN = f"{CONDA_PREFIX}/bin/conda"
CONDA_PY = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin/python"
CONDA_PIP = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin/pip"

DRIVE_CONDA_CACHE = "/content/drive/MyDrive/soulxflash_conda_cache/soulxflash_conda_env.tar"

if not os.path.exists(CONDA_PY):
    if os.path.exists(DRIVE_CONDA_CACHE):
        print("[캐시 발견] Drive에서 conda 환경 전체를 복원합니다 (몇 분 소요)...")
        os.makedirs(CONDA_PREFIX, exist_ok=True)
        !tar -xf "{DRIVE_CONDA_CACHE}" -C {CONDA_PREFIX}
    else:
        print("[캐시 없음] Miniconda 설치부터 시작합니다...")
        if not os.path.exists(CONDA_BIN):
            !wget -O /content/miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
            !bash /content/miniconda.sh -b -u -p {CONDA_PREFIX}
        !{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
        !{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
        !{CONDA_BIN} create -y -n {ENV_NAME} python=3.10
else:
    print("conda 환경이 이미 준비되어 있습니다:", CONDA_PY)

if os.path.exists(CONDA_PY):
    print("conda 환경 준비 완료:", CONDA_PY)
else:
    raise RuntimeError("conda 환경이 만들어지지 않았습니다. 위 로그를 보고 원인을 확인한 뒤 다시 실행하세요.")

# 노트북 커널(3.12) 쪽에서 필요한 것 (ngrok 터널용)
!pip install -q pyngrok


In [ ]:
# 3. SoulX-FlashHead 레포 클론 (최초 1회)
if not os.path.exists("/content/SoulX-FlashHead/generate_video.py"):
    !git clone -q https://github.com/Soul-AILab/SoulX-FlashHead.git /content/SoulX-FlashHead
else:
    print("이미 클론되어 있습니다.")


In [ ]:
# 4. 의존성 설치
# torch는 CUDA 12.8 wheel을 명시적으로 받아야 합니다 (flash_attn 빌드가 이 조합을 기준으로 합니다).
!{CONDA_PIP} install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu128 torch==2.7.1 torchvision==0.22.1

# requirements.txt를 그대로 설치하면 gradio/xformers/flask/nvidia-nccl까지 딸려 들어오는데,
# 우리 서버는 gradio/flask를 안 쓰고, xformers는 핵심 모델 코드(flash_head_model.py,
# flash_head_pipeline.py, vae.py) 어디에서도 안 쓰이는 걸 확인했습니다 (world_size=1로만
# 쓸 거라 분산 추론용 xfuser 경로도 안 탑니다). 이 네 개만 빼고 설치합니다.
EXCLUDE_PACKAGES = {"gradio", "xformers", "flask", "nvidia-nccl-cu12"}

with open("/content/SoulX-FlashHead/requirements.txt") as f:
    _req_lines = f.readlines()

_filtered_lines = [
    line for line in _req_lines
    if line.strip() and not any(line.strip().startswith(pkg) for pkg in EXCLUDE_PACKAGES)
]
with open("/content/soulx_requirements_filtered.txt", "w") as f:
    f.writelines(_filtered_lines)

!{CONDA_PIP} install -q -r /content/soulx_requirements_filtered.txt

# flash_attn은 attention 우선순위 체인(SageAttention > FlashAttention3 > FlashAttention2 > 기본)에서
# 실제로 속도에 영향을 주는 선택적 의존성이라, 이번엔 제대로 설치되게 시도합니다.
# 사전 빌드된 wheel을 먼저 시도하고, 실패하면 ninja를 깐 뒤 소스 빌드로 재시도합니다.
# 실패해도 셀이 죽지 않게 하고, 실패 원인을 볼 수 있도록 로그 마지막 부분을 출력합니다.
import subprocess as _sp

def _install_flash_attn():
    print("[flash_attn] 사전 빌드된 wheel 설치 시도...")
    result = _sp.run(
        [CONDA_PIP, "install", "flash_attn==2.8.0.post2"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print("[flash_attn] 사전 빌드된 wheel 설치 성공")
        return True

    print("[flash_attn] 사전 빌드된 wheel 실패 - 마지막 로그:")
    print((result.stdout + result.stderr)[-2000:])

    print("\n[flash_attn] ninja 설치 후 소스 빌드로 재시도 (오래 걸릴 수 있음)...")
    _sp.run([CONDA_PIP, "install", "-q", "ninja"], check=True)
    result = _sp.run(
        [CONDA_PIP, "install", "--no-build-isolation", "flash_attn==2.8.0.post2"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print("[flash_attn] 소스 빌드 성공")
        return True

    print("[flash_attn] 소스 빌드도 실패 - 마지막 로그:")
    print((result.stdout + result.stderr)[-3000:])
    print("[flash_attn] 설치를 포기하고 계속 진행합니다 (PyTorch 기본 attention으로 자동 대체됩니다).")
    return False

_flash_attn_ok = _install_flash_attn()

# 우리가 직접 짤 FastAPI 스트리밍 서버용 (SoulX 저장소 자체는 gradio만 쓰므로 별도 설치 필요)
!{CONDA_PIP} install -q fastapi==0.111.0 "uvicorn[standard]==0.30.1" websockets==15.0.1

print("\n설치 확인:")
!{CONDA_PY} -c "import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())"
if _flash_attn_ok:
    !{CONDA_PY} -c "import flash_attn; print('flash_attn', flash_attn.__version__)"


In [ ]:
# 4-1. conda 환경 전체를 Drive에 캐시
# 회사 계정이라 Drive 스토리지를 추가로 구매하기 어렵고, SoulX-FlashHead는 아직 테스트 단계라
# 일단 Drive 캐싱 없이 진행합니다 (매 런타임마다 처음부터 다시 설치해야 해서 느리지만, Drive 용량은 안 씁니다).
# 나중에 정식으로 쓰기로 하면 아래 CACHE_TO_DRIVE를 True로 바꾸면 됩니다.
CACHE_TO_DRIVE = False

if not CACHE_TO_DRIVE:
    print("CACHE_TO_DRIVE = False - Drive 캐싱을 건너뜁니다 (매 런타임마다 재설치됩니다).")
elif not os.path.exists(DRIVE_CONDA_CACHE):
    print("conda 환경을 Drive에 캐시합니다 (용량이 커서 몇 분 걸릴 수 있습니다)...")
    !tar -cf /content/soulxflash_conda_env.tar -C {CONDA_PREFIX} .
    os.makedirs(os.path.dirname(DRIVE_CONDA_CACHE), exist_ok=True)
    !cp /content/soulxflash_conda_env.tar "{DRIVE_CONDA_CACHE}"
    print("Drive 캐시 완료.")
else:
    print("conda 환경 캐시가 이미 Drive에 있습니다:", DRIVE_CONDA_CACHE)


In [ ]:
# 5. SoulX-FlashHead 체크포인트 다운로드 (지금은 CACHE_TO_DRIVE = False라 Drive에는 안 올림)
import subprocess as _sp

DRIVE_SOULX_MODELS = "/content/drive/MyDrive/soulx_flashhead_models"
LOCAL_MODELS_DIR = "/content/SoulX-FlashHead/models"

if os.path.exists(DRIVE_SOULX_MODELS) and os.listdir(DRIVE_SOULX_MODELS):
    print("[캐시 발견] Drive에서 모델을 복사합니다...")
    os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
    _sp.run(["cp", "-r", f"{DRIVE_SOULX_MODELS}/.", f"{LOCAL_MODELS_DIR}/"], check=True)
else:
    print("[캐시 없음] Hugging Face에서 다운로드합니다...")
    # huggingface_hub를 -U(최신)로 깔면 1.x가 설치되는데, 1.x부터는 huggingface-cli 명령이
    # 폐지되고(hf로 대체) transformers==4.57.3이 요구하는 huggingface-hub<1.0과도 충돌합니다.
    # transformers가 허용하는 범위 안에서 고정해서 옛 huggingface-cli 명령이 계속 동작하게 합니다.
    _sp.run(
        [CONDA_PIP, "install", "-q", "huggingface_hub[cli]>=0.34.0,<1.0"],
        check=True,
    )
    HF_CLI = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin/huggingface-cli"

    # SoulX-FlashHead-1_3B 저장소 전체(~13~20GB)에는 Lite 모델(~6.2GB) 외에
    # Model_Pro(~5.6GB, 안 씀 - dual RTX5090 전용), VAE_LTX(~1.6GB, 안 쓰는 파이프라인용),
    # assets(데모 영상)까지 들어있어서, 실제로 필요한 Model_Lite/VAE_Wan/설정 파일만 받습니다.
    _sp.run(
        [HF_CLI, "download", "Soul-AILab/SoulX-FlashHead-1_3B",
         "--include", "Model_Lite/*", "VAE_Wan/*", "config.json", "model_index.json",
         "--local-dir", f"{LOCAL_MODELS_DIR}/SoulX-FlashHead-1_3B"],
        check=True,
    )
    _sp.run(
        [HF_CLI, "download", "facebook/wav2vec2-base-960h",
         "--local-dir", f"{LOCAL_MODELS_DIR}/wav2vec2-base-960h"],
        check=True,
    )

    if CACHE_TO_DRIVE:
        os.makedirs(DRIVE_SOULX_MODELS, exist_ok=True)
        _sp.run(["cp", "-r", f"{LOCAL_MODELS_DIR}/.", f"{DRIVE_SOULX_MODELS}/"], check=True)
        print("Drive 캐시 완료.")
    else:
        print("CACHE_TO_DRIVE = False - Drive 캐싱을 건너뜁니다 (로컬 디스크에만 있습니다).")

print("모델 파일 확인:")
!du -sh {LOCAL_MODELS_DIR}/SoulX-FlashHead-1_3B {LOCAL_MODELS_DIR}/wav2vec2-base-960h
!ls -la {LOCAL_MODELS_DIR}/SoulX-FlashHead-1_3B {LOCAL_MODELS_DIR}/wav2vec2-base-960h


In [ ]:
# 6. 배경/조건 이미지 준비
# MuseTalk 노트북과 동일한 원본 듀오 영상을 재사용합니다 (같은 위치에 이미 업로드돼 있어야 함).
# SoulX-FlashHead는 항상 512x512 정사각형으로만 생성하기 때문에, 얼굴을 OpenCV로 자동 검출해서
# 그 주변 512x512만 정확히 크롭한 걸 조건 이미지로 씁니다 (화질 우선 조합: use_face_crop=True와
# 같이 쓰면 SoulX가 여기서 한 번 더 확대해서 생성하기 때문에, 저희가 계산한 이 512x512 위치와
# 실제 생성된 얼굴의 확대/위치가 정확히는 안 맞습니다 - TODO 참고).
#
# TODO(SoulX 포지셔닝 보정, 나중에 시간 되면): use_face_crop=True일 때 SoulX가 내부적으로
# 얼마나 더 확대하는지 배율을 아직 실측 안 했습니다. 한 번 생성해서 결과 프레임에서 얼굴이
# 원본 대비 몇 배 커졌는지 재본 뒤, 8번 셀 _writer()에서 그 배율의 역수만큼 축소해서 배치하도록
# 보정값을 하드코딩해야 위치가 정확히 맞습니다. 지금은 미보정 상태로 둡니다.
import subprocess, json as _json
import cv2 as _cv2

DUO_SOURCE_PATH = "/content/drive/MyDrive/musetalk_assets/avatar_duo.mp4"
assert os.path.exists(DUO_SOURCE_PATH), f"원본 영상을 찾을 수 없습니다: {DUO_SOURCE_PATH}"

os.makedirs("/content/SoulX-FlashHead/data", exist_ok=True)

DUO_SOURCE_VIDEO_25FPS = "/content/SoulX-FlashHead/data/avatar_duo_25fps.mp4"
!ffmpeg -y -i "$DUO_SOURCE_PATH" -r 25 "$DUO_SOURCE_VIDEO_25FPS"


def get_video_size(path):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height", "-of", "json", path],
        capture_output=True, text=True, check=True,
    )
    stream = _json.loads(result.stdout)["streams"][0]
    return stream["width"], stream["height"]


duo_width, duo_height = get_video_size(DUO_SOURCE_VIDEO_25FPS)
print(f"듀오 영상 해상도: {duo_width}x{duo_height}")

DUO_BACKGROUND_IMAGE = "/content/SoulX-FlashHead/data/avatar_duo_background.png"
!ffmpeg -y -i "$DUO_SOURCE_VIDEO_25FPS" -vframes 1 "$DUO_BACKGROUND_IMAGE"

# 얼굴 위치 탐색용으로 0초(전환 중일 수 있음)가 아니라 1초 지점 프레임을 사용합니다.
SAMPLE_FRAME_PATH = "/content/SoulX-FlashHead/data/avatar_duo_sample_1s.png"
!ffmpeg -y -ss 1 -i "$DUO_SOURCE_VIDEO_25FPS" -vframes 1 "$SAMPLE_FRAME_PATH"
_sample_frame = _cv2.imread(SAMPLE_FRAME_PATH)

# 왼쪽/오른쪽 절반 (왼쪽=personality, 오른쪽=technical) - 얼굴 검출 대상 영역을 좁히는 용도로만 씁니다.
HALF_REGIONS = {
    "technical":   {"x": duo_width // 2, "y": 0, "w": duo_width - duo_width // 2, "h": duo_height},
    "personality": {"x": 0,              "y": 0, "w": duo_width // 2,            "h": duo_height},
}

FACE_BOX_SIZE = 512
_face_cascade = _cv2.CascadeClassifier(_cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

FACE_REGIONS = {}
for avatar_type, half in HALF_REGIONS.items():
    half_img = _sample_frame[half["y"]:half["y"] + half["h"], half["x"]:half["x"] + half["w"]]
    gray = _cv2.cvtColor(half_img, _cv2.COLOR_BGR2GRAY)
    faces = _face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))

    if len(faces) > 0:
        # 가장 큰 얼굴을 기준으로 삼습니다 (여러 개 검출될 경우 대비).
        fx, fy, fw, fh = max(faces, key=lambda f: f[2] * f[3])
        center_x, center_y = fx + fw // 2, fy + fh // 2
        print(f"[{avatar_type}] 얼굴 검출 성공: 절반 영역 기준 중심 ({center_x}, {center_y})")
    else:
        # 검출 실패 시 절반 영역의 가로 중앙, 세로 35% 지점(보통 얼굴이 있는 위치)으로 대체합니다.
        center_x, center_y = half["w"] // 2, int(half["h"] * 0.35)
        print(f"[{avatar_type}] 얼굴 검출 실패 - 기본 위치로 대체: ({center_x}, {center_y})")

    # 512x512 박스를 얼굴 중심에 맞추되, 절반 영역(그리고 전체 프레임) 밖으로 안 나가게 clamp합니다.
    local_x = max(0, min(center_x - FACE_BOX_SIZE // 2, half["w"] - FACE_BOX_SIZE))
    local_y = max(0, min(center_y - FACE_BOX_SIZE // 2, half["h"] - FACE_BOX_SIZE))

    FACE_REGIONS[avatar_type] = {
        "x": half["x"] + local_x,
        "y": half["y"] + local_y,
        "w": FACE_BOX_SIZE,
        "h": FACE_BOX_SIZE,
    }

print("얼굴 기준 512x512 크롭 영역 (전체 프레임 좌표):", FACE_REGIONS)

with open("/content/SoulX-FlashHead/data/crop_regions.json", "w") as f:
    _json.dump(FACE_REGIONS, f)

COND_IMAGES = {}
for avatar_type, region in FACE_REGIONS.items():
    crop_filter = f"crop={region['w']}:{region['h']}:{region['x']}:{region['y']}"
    cond_path = f"/content/SoulX-FlashHead/data/avatar_duo_{avatar_type}_cond.png"
    !ffmpeg -y -ss 1 -i "$DUO_SOURCE_VIDEO_25FPS" -vf "{crop_filter}" -vframes 1 "$cond_path"
    COND_IMAGES[avatar_type] = cond_path

print("SoulX-FlashHead 조건 이미지:", COND_IMAGES)


In [ ]:
# 7. SoulX-FlashHead 듀오 서버 스크립트 작성
# 구조는 MuseTalk 듀오 서버(musetalk_server.py)와 동일합니다 - ffmpeg로 배경 위에 오버레이 합성하며
# fMP4를 실시간 스트리밍하는 부분은 그대로 재사용하고, 프레임을 만드는 내부 로직만
# SoulX-FlashHead(flash_head.inference)로 교체했습니다.
server_script = '\nimport os, sys, glob, time, types, shutil, subprocess, base64, uuid, json\nimport queue as _queue\nimport threading as _threading\nfrom collections import deque\n\nimport numpy as np\nimport cv2\nimport torch\nimport librosa\n\nfrom fastapi import FastAPI, Request\nfrom fastapi.responses import StreamingResponse\nimport uvicorn\n\nsys.path.insert(0, "/content/SoulX-FlashHead")\nfrom flash_head.inference import (\n    get_pipeline,\n    get_base_data,\n    get_infer_params,\n    get_audio_embedding,\n    run_pipeline,\n)\n\nCKPT_DIR = "/content/SoulX-FlashHead/models/SoulX-FlashHead-1_3B"\nWAV2VEC_DIR = "/content/SoulX-FlashHead/models/wav2vec2-base-960h"\n# A100/L4 단일 GPU 실시간이 목표라 Lite로 시작합니다 (Pro는 dual RTX5090이 있어야 실시간).\nMODEL_TYPE = "lite"\n\nCOND_IMAGES = {\n    "technical": "/content/SoulX-FlashHead/data/avatar_duo_technical_cond.png",\n    "personality": "/content/SoulX-FlashHead/data/avatar_duo_personality_cond.png",\n}\n\nwith open("/content/SoulX-FlashHead/data/crop_regions.json") as f:\n    CROP_REGIONS = json.load(f)\n\nDUO_BACKGROUND_IMAGE = "/content/SoulX-FlashHead/data/avatar_duo_background.png"\nDUO_IDLE_LOOP_VIDEO = "/content/SoulX-FlashHead/data/avatar_duo_25fps.mp4"\n\n_probe_result = subprocess.run(\n    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", DUO_IDLE_LOOP_VIDEO],\n    capture_output=True, text=True, check=True,\n)\nDUO_IDLE_LOOP_DURATION = float(_probe_result.stdout.strip())\n\n_bg_probe = cv2.imread(DUO_BACKGROUND_IMAGE)\nduo_height, duo_width = _bg_probe.shape[:2]\n\n# 배경 루프 재생 위치는 두 아바타가 공유하는 하나의 값으로 관리합니다 (MuseTalk 서버와 동일한 패턴).\n_background_position_seconds = [0.0]\n\nprint("SoulX-FlashHead 파이프라인 로딩 중 (아바타별로 독립된 인스턴스를 만듭니다)...", flush=True)\n\n# 아바타 2명이 파이프라인/오디오 캐시를 각자 따로 가지도록 설계했습니다.\n# 하나의 파이프라인을 공유하면서 매 요청마다 get_base_data로 조건 이미지를 바꿔치기하는 방식은\n# 동시 요청이 들어왔을 때 서로 상태를 덮어쓰는 문제가 생길 수 있어(이전에 MuseTalk 서버에서\n# 겪었던 전역 상태 충돌과 같은 종류의 문제) 처음부터 아바타별로 완전히 분리했습니다.\npipelines = {}\ninfer_params = None\nfor _avatar_type, _cond_image in COND_IMAGES.items():\n    print(f"[{_avatar_type}] 파이프라인 로드 시작", flush=True)\n    _pipeline = get_pipeline(\n        world_size=1,\n        ckpt_dir=CKPT_DIR,\n        model_type=MODEL_TYPE,\n        wav2vec_dir=WAV2VEC_DIR,\n    )\n    # 조건 이미지가 이미 얼굴 중심 512x512로 타이트하게 크롭돼 있어서(더 자를 여지가 거의 없음),\n    # use_face_crop=True로 SoulX 자체 얼굴 정렬/전처리를 태우면 프레이밍은 거의 그대로 유지되면서\n    # 모델이 학습 때 기대하는 전처리 방식과 맞아떨어져 화질이 더 좋게 나오는 것을 확인했습니다.\n    get_base_data(_pipeline, cond_image_path_or_dir=_cond_image, base_seed=9999, use_face_crop=True)\n    pipelines[_avatar_type] = _pipeline\n    if infer_params is None:\n        infer_params = get_infer_params()\n    print(f"[{_avatar_type}] 파이프라인 준비 완료", flush=True)\n\nSAMPLE_RATE = infer_params["sample_rate"]\nTGT_FPS = infer_params["tgt_fps"]\nCACHED_AUDIO_DURATION = infer_params["cached_audio_duration"]\nFRAME_NUM = infer_params["frame_num"]\nMOTION_FRAMES_NUM = infer_params["motion_frames_num"]\nSLICE_LEN = FRAME_NUM - MOTION_FRAMES_NUM\nSLICE_LEN_SAMPLES = SLICE_LEN * SAMPLE_RATE // TGT_FPS\nCACHED_AUDIO_LENGTH_SUM = SAMPLE_RATE * CACHED_AUDIO_DURATION\nAUDIO_END_IDX = CACHED_AUDIO_DURATION * TGT_FPS\nAUDIO_START_IDX = AUDIO_END_IDX - FRAME_NUM\n\nprint(\n    f"infer_params: sample_rate={SAMPLE_RATE} tgt_fps={TGT_FPS} frame_num={FRAME_NUM} "\n    f"motion_frames_num={MOTION_FRAMES_NUM}",\n    flush=True,\n)\n\n# 아바타별 오디오 롤링 버퍼. gradio_app_streaming.py의 audio_dq와 같은 역할이지만,\n# 원본은 요청 1번에 오디오 전체를 처리하는 구조라 이 deque가 함수 지역 변수였습니다.\n# 우리는 문장 단위로 여러 번 나눠서 스트리밍 요청이 들어오기 때문에, 같은 아바타의 이전\n# 문장이 끝난 지점에서 이어지도록 프로세스 전역에 아바타별로 유지합니다.\n# (주의: 같은 아바타에 대한 요청이 동시에 두 개 이상 들어오면 이 버퍼가 꼬일 수 있습니다 -\n#  프런트엔드가 한 아바타에 순차적으로만 요청한다는 전제입니다.)\n_audio_deques = {\n    _avatar_type: deque([0.0] * CACHED_AUDIO_LENGTH_SUM, maxlen=CACHED_AUDIO_LENGTH_SUM)\n    for _avatar_type in COND_IMAGES\n}\n\n_frame_shape_logged = set()\n\n# TODO(SoulX 포지셔닝 보정, 나중에 시간 되면): use_face_crop=True일 때 SoulX가 조건 이미지에서\n# 얼굴을 한 번 더 확대해서 생성하기 때문에, 지금은 512x512 조건 이미지 위치와 실제 생성된\n# 얼굴 위치/크기가 정확히 안 맞습니다. 배율을 실측해서 아래 값을 채우면\n# _resize_cover 대신(또는 이전에) 이 배율의 역수만큼 축소해서 중앙 배치하도록 보정할 수 있습니다.\nSOULX_FACE_ZOOM_COMPENSATION = 1.0  # 1.0 = 미보정. 실측 후 이 값을 조정하세요.\n\n\ndef _resize_cover(img, target_w, target_h):\n    """\n    종횡비를 유지한 채 목표 크기를 빈틈없이 채우도록 리사이즈한 뒤 가운데를 크롭합니다\n    (CSS object-fit: cover와 동일). SoulX가 생성하는 프레임의 종횡비가 듀오 프레임의\n    절반(크롭 영역) 종횡비와 달라서, 단순 stretch resize를 쓰면 이미지가 눌리거나\n    늘어나 보이는 문제가 있어 이렇게 바꿨습니다.\n    """\n    h, w = img.shape[:2]\n    scale = max(target_w / w, target_h / h)\n    new_w, new_h = round(w * scale), round(h * scale)\n    resized = cv2.resize(img, (new_w, new_h))\n    x_start = (new_w - target_w) // 2\n    y_start = (new_h - target_h) // 2\n    return resized[y_start:y_start + target_h, x_start:x_start + target_w]\n\n\ndef _generate_chunks(avatar_type, audio_path):\n    """\n    audio_path를 SLICE_LEN 프레임 단위 청크로 나눠서 run_pipeline을 호출하고,\n    motion_frames_num을 제외한 새 프레임(numpy, RGB)을 청크마다 yield합니다.\n    Soul-AILab의 gradio_app_streaming.py: inference_worker()를 이식한 버전입니다.\n    """\n    pipeline = pipelines[avatar_type]\n    audio_dq = _audio_deques[avatar_type]\n\n    human_speech_array, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)\n\n    remainder = len(human_speech_array) % SLICE_LEN_SAMPLES\n    if remainder > 0:\n        pad_length = SLICE_LEN_SAMPLES - remainder\n        human_speech_array = np.concatenate(\n            [human_speech_array, np.zeros(pad_length, dtype=human_speech_array.dtype)]\n        )\n    slices = human_speech_array.reshape(-1, SLICE_LEN_SAMPLES)\n\n    for chunk_idx, chunk_audio in enumerate(slices):\n        audio_dq.extend(chunk_audio.tolist())\n        audio_array = np.array(audio_dq)\n        audio_embedding = get_audio_embedding(pipeline, audio_array, AUDIO_START_IDX, AUDIO_END_IDX)\n        torch.cuda.synchronize()\n        _chunk_t0 = time.time()\n        video = run_pipeline(pipeline, audio_embedding)\n        video = video[MOTION_FRAMES_NUM:]\n        torch.cuda.synchronize()\n        print(\n            f"[batch-timing][{avatar_type}] 청크{chunk_idx} 추론 {time.time() - _chunk_t0:.3f}s "\n            f"(프레임 {video.shape[0]}개, 실시간예산 {video.shape[0] / TGT_FPS:.3f}s)",\n            flush=True,\n        )\n        yield video.cpu().numpy()\n\n\ndef _warmup(avatar_type):\n    """\n    torch.compile(TorchInductor) 최초 실행 시 커널을 새로 컴파일하는 비용이 첫 번째 실제\n    요청에서 발생하지 않도록, 서버 기동 시 무음 오디오로 한 번 미리 추론을 돌려둡니다.\n    (MuseTalk 서버의 _warmup_batch_sizes와 같은 목적 - 실측 로그에서 첫 청크만 177초,\n    이후 청크는 0.25초로 떨어지는 것을 확인했습니다.)\n    """\n    dummy_audio_path = f"/content/_warmup_silence_{avatar_type}.wav"\n    subprocess.run(\n        ["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=r=16000:cl=mono", "-t", "2", dummy_audio_path],\n        check=True, capture_output=True,\n    )\n    print(f"[{avatar_type}] 워밍업 시작 (최초 컴파일 비용을 서버 기동 시점으로 옮깁니다)...", flush=True)\n    _warmup_t0 = time.time()\n    for _ in _generate_chunks(avatar_type, dummy_audio_path):\n        pass\n    print(f"[{avatar_type}] 워밍업 완료 ({time.time() - _warmup_t0:.1f}초)", flush=True)\n    os.remove(dummy_audio_path)\n\n\nfor _avatar_type in pipelines:\n    _warmup(_avatar_type)\n\nprint("SoulX-FlashHead 듀오 아바타 준비 완료:", list(pipelines.keys()), flush=True)\n\n\ndef duo_stream_inference(avatar_type, audio_path):\n    """\n    지정된 아바타 한 명이 audio_path를 말하는 모습을, 듀오 배경 위 원래 자리에\n    실시간으로 겹쳐 합성하면서 fMP4 청크를 yield하는 제너레이터.\n    ffmpeg 파이프라인 구조는 MuseTalk 듀오 서버(musetalk_server.py)와 동일합니다 -\n    프레임을 만드는 내부 로직만 SoulX-FlashHead로 바뀌었습니다.\n    """\n    _t0 = time.time()\n    print(f"[stream-timing][{avatar_type}] duo_stream_inference 시작", flush=True)\n\n    region = CROP_REGIONS[avatar_type]\n    start_bg_offset = _background_position_seconds[0] % DUO_IDLE_LOOP_DURATION\n\n    cmd = [\n        "ffmpeg", "-y", "-v", "quiet", "-fflags", "+genpts",\n        "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{region[\'w\']}x{region[\'h\']}", "-r", str(TGT_FPS),\n        "-thread_queue_size", "512", "-i", "pipe:0",\n        "-ss", f"{start_bg_offset:.3f}", "-stream_loop", "-1", "-i", DUO_IDLE_LOOP_VIDEO,\n        "-thread_queue_size", "512", "-i", audio_path,\n        "-filter_complex",\n        f"[1:v]scale={duo_width}:{duo_height}[bg];"\n        f"[bg][0:v]overlay={region[\'x\']}:{region[\'y\']}:shortest=1,format=yuv420p[outv]",\n        "-map", "[outv]", "-map", "2:a",\n        # 청크당 추론이 실시간 예산 대비 여유(약 4배)가 있어서, 인코딩 속도를 좀 더\n        # 화질 쪽으로 돌려도 실시간에는 지장이 없을 것 같아 preset/crf를 화질 우선으로 올립니다.\n        "-c:v", "libx264", "-preset", "fast", "-tune", "zerolatency", "-crf", "18",\n        "-g", "12", "-sc_threshold", "0",\n        "-c:a", "aac", "-b:a", "128k",\n        "-movflags", "frag_keyframe+empty_moov+default_base_moof",\n        "-shortest", "-f", "mp4", "pipe:1",\n    ]\n    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)\n\n    output_q = _queue.Queue()\n    _first_chunk_logged = [False]\n\n    def _read_stdout():\n        try:\n            while True:\n                chunk = proc.stdout.read(8192)\n                if not chunk:\n                    break\n                if not _first_chunk_logged[0]:\n                    print(f"[stream-timing][{avatar_type}] ffmpeg 첫 출력 청크: {time.time() - _t0:.2f}초", flush=True)\n                    _first_chunk_logged[0] = True\n                output_q.put(chunk)\n        finally:\n            output_q.put(None)\n\n    reader = _threading.Thread(target=_read_stdout, daemon=True)\n    reader.start()\n\n    def _writer():\n        total_frames = 0\n        try:\n            # _generate_chunks는 모델 입력용으로 오디오를 청크 크기 배수까지 패딩해서 처리하기\n            # 때문에, 원본(패딩 전) 오디오 길이보다 프레임을 더 많이 만들어낼 수 있습니다.\n            # ffmpeg에는 패딩 안 된 원본 오디오를 그대로 넣고 -shortest를 쓰고 있어서,\n            # 원본 오디오가 먼저 끝나 ffmpeg가 먼저 종료되면 남은 패딩분 프레임을 쓰다가\n            # Broken pipe가 났습니다. 원본 오디오 길이만큼만 쓰고 스스로 멈추도록 프레임 수를 제한합니다.\n            _probe_y, _probe_sr = librosa.load(audio_path, sr=None)\n            max_frames = round(len(_probe_y) / _probe_sr * TGT_FPS)\n\n            for frames_np in _generate_chunks(avatar_type, audio_path):\n                for frame in frames_np:\n                    if total_frames >= max_frames:\n                        break\n                    if avatar_type not in _frame_shape_logged:\n                        print(f"[{avatar_type}] SoulX 원본 생성 프레임 크기: {frame.shape}, 목표 크롭 영역: {region[\'w\']}x{region[\'h\']}", flush=True)\n                        _frame_shape_logged.add(avatar_type)\n                    # SoulX-FlashHead 출력은 RGB라, ffmpeg 입력 포맷(bgr24)에 맞춰 채널을 뒤집습니다.\n                    bgr = cv2.cvtColor(frame.astype(np.uint8), cv2.COLOR_RGB2BGR)\n                    resized = _resize_cover(bgr, region["w"], region["h"])\n                    proc.stdin.write(resized.tobytes())\n                    total_frames += 1\n                if total_frames >= max_frames:\n                    break\n        except Exception as e:\n            print(f"[duo-stream][{avatar_type}] 생성 오류: {e}", flush=True)\n        finally:\n            _background_position_seconds[0] += total_frames / TGT_FPS\n            try:\n                proc.stdin.close()\n            except Exception:\n                pass\n\n    writer = _threading.Thread(target=_writer, daemon=True)\n    writer.start()\n\n    while True:\n        chunk = output_q.get()\n        if chunk is None:\n            break\n        yield chunk\n\n    writer.join(timeout=5)\n    proc.wait(timeout=5)\n\n\napp = FastAPI()\n\n\n@app.post("/synthesize/duo/stream")\nasync def duo_stream_handler(request: Request):\n    """\n    아바타 한 명이 오디오 하나를 말하는 실시간 스트리밍 전용 엔드포인트.\n    MuseTalk 듀오 서버의 같은 이름 엔드포인트와 요청/응답 형식을 동일하게 맞췄습니다.\n    """\n    body = await request.json()\n    avatar_type = body.get("avatar_type", "technical")\n    audio_base64 = body.get("audio_base64")\n\n    if avatar_type not in pipelines:\n        return {"error": f"알 수 없는 avatar_type: {avatar_type}"}\n    if not audio_base64:\n        return {"error": "audio_base64가 필요합니다."}\n\n    audio_bytes = base64.b64decode(audio_base64)\n    audio_path = f"/content/tmp_soulx_stream_{avatar_type}_{uuid.uuid4().hex}.wav"\n    with open(audio_path, "wb") as f:\n        f.write(audio_bytes)\n\n    def generate():\n        try:\n            for chunk in duo_stream_inference(avatar_type, audio_path):\n                yield chunk\n        finally:\n            if os.path.exists(audio_path):\n                os.remove(audio_path)\n\n    return StreamingResponse(\n        generate(),\n        media_type="video/mp4",\n        headers={"Cache-Control": "no-cache", "X-Content-Type-Options": "nosniff"},\n    )\n\n\nif __name__ == "__main__":\n    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")\n'

with open("/content/SoulX-FlashHead/soulx_server.py", "w", encoding="utf-8") as f:
    f.write(server_script)

print("soulx_server.py 작성 완료 (듀오 아바타 SoulX-FlashHead 스트리밍 서버)")


## 서버 프로세스 실행

아래 셀은 `soulx_server.py`를 conda 환경(`soulxflash`)의 파이썬으로 서브프로세스 실행합니다.
파이프라인 로딩(아바타 2개 각각 별도 인스턴스) 때문에 처음 기동은 시간이 걸릴 수 있습니다.


In [ ]:
# 8. 서버 프로세스 실행 + 기동 대기
import subprocess, time

if "server_proc" in globals() and server_proc.poll() is None:
    print("기존 서버 프로세스 종료 중...")
    server_proc.kill()
    server_proc.wait()

log_path = "/content/soulx_server.log"
log_file = open(log_path, "w")

server_env = dict(os.environ)
server_env["MPLBACKEND"] = "Agg"

server_proc = subprocess.Popen(
    [CONDA_PY, "/content/SoulX-FlashHead/soulx_server.py"],
    cwd="/content/SoulX-FlashHead",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=server_env,
)
print(f"서버 프로세스 시작 (PID {server_proc.pid})")

content = ""
ready = False
for _ in range(180):
    with open(log_path) as f:
        content = f.read()
    if "Uvicorn running on" in content:
        ready = True
        break
    if server_proc.poll() is not None:
        print("서버 프로세스가 예기치 않게 종료되었습니다. 아래 로그를 확인하세요.")
        break
    time.sleep(5)

print("서버 기동 완료!" if ready else "서버 기동되지 않았습니다 - 이 셀을 다시 실행해서 계속 대기하거나 로그를 확인하세요.")
print("----- 최근 로그 -----")
print(content[-3000:])


## 실시간 스트리밍 헬스체크

MuseTalk 노트북과 동일한 `/synthesize/duo/stream` 엔드포인트로 짧은 문장을 실제로 합성해봅니다.


In [ ]:
# 9. 실시간 스트리밍 헬스체크
!pip install -q edge-tts
import base64, requests

EVAL_SENTENCES = [
    ("technical", "안녕하세요, 답변 감사합니다."),
    ("personality", "네, 다시 인사드릴게요."),
]

for avatar_type, text in EVAL_SENTENCES:
    mp3_path = f"/content/soulx_health_{avatar_type}.mp3"
    wav_path = f"/content/soulx_health_{avatar_type}.wav"
    !edge-tts --voice ko-KR-SunHiNeural --text "{text}" --write-media {mp3_path}
    !ffmpeg -y -i {mp3_path} {wav_path}

    with open(wav_path, "rb") as f:
        audio_b64 = base64.b64encode(f.read()).decode("utf-8")

    print(f"생성 요청 중: {avatar_type} ...")
    t0 = __import__("time").time()
    resp = requests.post(
        "http://localhost:8000/synthesize/duo/stream",
        json={"avatar_type": avatar_type, "audio_base64": audio_b64},
        timeout=180,
    )
    elapsed = __import__("time").time() - t0

    if resp.status_code != 200 or len(resp.content) < 1000:
        print(f"  실패 (status={resp.status_code}, size={len(resp.content)}): {resp.text[:300]}")
        continue

    dest = f"/content/soulx_health_{avatar_type}_out.mp4"
    with open(dest, "wb") as f:
        f.write(resp.content)
    print(f"  완료: {dest} ({len(resp.content)/1024:.1f}KB, {elapsed:.2f}초)")


In [ ]:
# 로그 확인
!tail -n 80 /content/soulx_server.log


In [ ]:
# 10. ngrok 터널 연결 (테스트용 - 매번 랜덤 URL, 고정 도메인 불필요)
# ngrok 무료 플랜은 계정당 고정(개발용) 도메인을 1개만 주는데, MuseTalk 노트북이 이미 쓰고
# 있어서 지금은 별도 고정 도메인 없이 랜덤 URL로 테스트합니다. 실행할 때마다 URL이 바뀝니다.
from pyngrok import ngrok, conf
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
conf.get_default().auth_token = NGROK_AUTHTOKEN

public_tunnel = ngrok.connect(8000)
print("SoulX-FlashHead 듀오 서버 공개 URL (테스트용, 재실행 시 바뀜):", public_tunnel.public_url)


## 알려진 리스크 / 확인이 필요한 가정

실제로 A100/L4에서 돌려보기 전에는 확인할 수 없어서, 첫 실행에서 문제가 생기면 아래부터 의심하면 됩니다.

1. **flash_attn 설치**: torch 2.7.1 + CUDA 12.8 조합의 사전 빌드 wheel이 없으면 소스 빌드로 넘어가는데, 이 경우 매우 오래 걸리거나 실패할 수 있습니다.
2. **아바타별 파이프라인 상태 유지**: `get_base_data`를 서버 기동 시 한 번만 호출하고, 이후 문장마다 `run_pipeline`을 계속 이어서 호출하는 방식으로 짰습니다. SoulX-FlashHead 파이프라인 객체가 내부적으로 이전 청크의 "모션 프레임" 상태를 정말로 유지해주는지는 원본 코드를 읽고 추정한 것이라, 문장과 문장 사이에서 아바타가 부자연스럽게 끊기면 이 가정을 의심해야 합니다.
3. **색상 채널(RGB/BGR)**: SoulX 출력이 RGB라고 가정하고 `cv2.COLOR_RGB2BGR` 변환을 넣었는데, 실제 색이 이상하게(파랗게/붉게) 나오면 이 변환이 반대로 필요하거나 불필요한 것일 수 있습니다.
4. **FPS 불일치**: 배경 루프 영상은 25fps로 만들었는데, SoulX-FlashHead의 `tgt_fps`가 25가 아니면 배경과 아바타 움직임 속도가 안 맞을 수 있습니다. 서버 로그의 `infer_params: ... tgt_fps=...` 출력으로 확인 가능합니다.
5. **실시간 여부**: 청크 하나 추론 시간(`[batch-timing]` 로그)이 그 청크의 실시간 예산보다 크면 버퍼링이 생깁니다 - A100/L4가 논문 기준 GPU(4090/5090)와 다르므로 실제로 재봐야 압니다.
